# Notebook 01: Notebook 00 Checkpoint CCDifference Regression

This notebook starts from the frozen Notebook 00 normalized/log1p `.h5ad` checkpoints for `cellranger_filtered_manual_ec_div30_core_samples_freeze` and builds the Notebook 01 analysis using `CCDifference` as the cell-cycle regression covariate.

Context already completed before this run:

1. Notebook 00 saved the frozen combined and per-sample AnnData checkpoints under `results/notebook00/cellranger_filtered_manual_ec_div30_core_samples_freeze/h5ad/`.
2. The cell-cycle diagnostic pass selected the Regev/Tirosh S and G2M gene lists present in the data, scored `S_score`, `G2M_score`, `phase`, and `CCDifference = S_score - G2M_score`, and saved before/after CCDifference PCA diagnostics under this run folder.
3. This Notebook 01 run then uses `CCDifference` as the actual covariate for `sc.pp.regress_out` in the main HVG/PCA/UMAP branch analysis.

The two branches compared in every scope are:

- `not_regressed`: no covariates regressed from `.X`; used as the baseline comparison.
- `regressed_ccdifference`: `sc.pp.regress_out(adata, ["CCDifference"])`; this is the final branch saved as `.h5ad`.

The comparison is run once on the combined object and once for each `run_sample_id`. Raw counts remain in `.layers["counts"]`; the frozen Notebook 00 checkpoints are read-only inputs and are not modified by this notebook.


## Imports And Configuration


In [ ]:
from pathlib import Path
import gc
import os
import sys

import pandas as pd
import scanpy as sc


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name)
    return default if raw is None or not raw.strip() else int(raw)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name)
    return default if raw is None or not raw.strip() else float(raw)


for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "python_notebooks" / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break

from mge_organoid_python.cell_cycle import (
    CCDIFFERENCE_KEY,
    cell_cycle_score_summary,
    score_cell_cycle_and_ccdifference,
    select_cell_cycle_genes,
)
from mge_organoid_python.data_sources import find_repo_root, resolve_data_root
from mge_organoid_python.notebook01_workflow import (
    Notebook01EmbeddingSettings,
    Notebook01HVGSettings,
    Notebook01InputPaths,
    Notebook01InputSettings,
    Notebook01OutputPaths,
    Notebook01RegressionVariant,
    Notebook01RunSettings,
    embedding_table,
    infer_run_sample_ids,
    parse_csv,
    planned_analysis_table,
    run_hvg_selection,
    run_regression_embedding_branch,
    save_pca_variance_plot,
    save_umap_plot,
    settings_to_frame,
    validate_notebook01_input,
)

sc.settings.verbosity = 2


In [ ]:
REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = resolve_data_root()

NOTEBOOK00_RUN_LABEL = os.environ.get(
    "NOTEBOOK01_NOTEBOOK00_RUN_LABEL",
    "cellranger_filtered_manual_ec_div30_core_samples_freeze",
)
RUN_LABEL = os.environ.get(
    "NOTEBOOK01_RUN_LABEL",
    f"{NOTEBOOK00_RUN_LABEL}_ccdifference_v1",
)
SCOPES = parse_csv(os.environ.get("NOTEBOOK01_SCOPES"), default=("combined", "per_sample"))
REGRESS_KEYS = parse_csv(os.environ.get("NOTEBOOK01_REGRESS_KEYS"), default=(CCDIFFERENCE_KEY,))
N_TOP_GENES = env_int("NOTEBOOK01_N_TOP_GENES", 2000)
N_PCS = env_int("NOTEBOOK01_N_PCS", 50)
N_NEIGHBORS = env_int("NOTEBOOK01_N_NEIGHBORS", 15)
LEIDEN_RESOLUTION = env_float("NOTEBOOK01_LEIDEN_RESOLUTION", 0.5)
RANDOM_STATE = env_int("NOTEBOOK01_RANDOM_STATE", 0)
REGRESS_N_JOBS = env_int("NOTEBOOK01_REGRESS_N_JOBS", 8)
SHOW_PLOTS = env_bool("NOTEBOOK01_SHOW_PLOTS", False)
SAVE_PLOTS = env_bool("NOTEBOOK01_SAVE_PLOTS", True)
WRITE_BRANCH_H5AD = env_bool("NOTEBOOK01_WRITE_BRANCH_H5AD", True)
WRITE_BRANCH_H5AD_BRANCHES = parse_csv(
    os.environ.get("NOTEBOOK01_WRITE_BRANCH_H5AD_BRANCHES"),
    default=("regressed_ccdifference",),
)

input_settings = Notebook01InputSettings(notebook00_run_label=NOTEBOOK00_RUN_LABEL)
run_settings = Notebook01RunSettings(run_label=RUN_LABEL, scopes=SCOPES)
hvg_settings = Notebook01HVGSettings(n_top_genes=N_TOP_GENES)
embedding_settings = Notebook01EmbeddingSettings(
    n_pcs=N_PCS,
    n_neighbors=N_NEIGHBORS,
    leiden_resolution=LEIDEN_RESOLUTION,
    random_state=RANDOM_STATE,
    regress_n_jobs=REGRESS_N_JOBS,
)
variants = (
    Notebook01RegressionVariant.not_regressed(),
    Notebook01RegressionVariant.ccdifference_regressed(regress_keys=REGRESS_KEYS),
)
input_paths = Notebook01InputPaths.from_data_root(DATA_ROOT, settings=input_settings)
output_paths = Notebook01OutputPaths.from_data_root(DATA_ROOT, settings=run_settings)
output_paths.ensure_dirs()

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("NOTEBOOK00_RUN_LABEL:", NOTEBOOK00_RUN_LABEL)
print("RUN_LABEL:", RUN_LABEL)
print("SCOPES:", SCOPES)
print("REGRESS_KEYS:", REGRESS_KEYS)
print("N_TOP_GENES:", N_TOP_GENES)
print("SHOW_PLOTS:", SHOW_PLOTS)
print("SAVE_PLOTS:", SAVE_PLOTS)
print("WRITE_BRANCH_H5AD:", WRITE_BRANCH_H5AD)
print("WRITE_BRANCH_H5AD_BRANCHES:", WRITE_BRANCH_H5AD_BRANCHES)
print("Notebook 00 input:", input_paths.combined_normalized_log1p)
print("Notebook 01 output:", output_paths.run_dir)

pd.concat(
    [
        settings_to_frame(input_settings, "notebook01_input"),
        settings_to_frame(run_settings, "notebook01_run"),
        settings_to_frame(hvg_settings, "notebook01_hvg"),
        settings_to_frame(embedding_settings, "notebook01_embedding"),
    ],
    ignore_index=True,
).to_csv(output_paths.table_dir / "notebook01_run_parameters.tsv", sep="	", index=False)


## Load Combined Checkpoint And Build Plan


In [ ]:
combined_adata = sc.read_h5ad(input_paths.combined_normalized_log1p)
combined_validation = validate_notebook01_input(
    combined_adata,
    input_path=input_paths.combined_normalized_log1p,
    counts_layer=input_settings.counts_layer,
    scope="combined",
)
if not combined_validation["notebook01_input_validation_passed"]:
    raise AssertionError(f"Combined Notebook 01 input validation failed: {combined_validation}")

run_sample_ids = infer_run_sample_ids(combined_adata)
analysis_plan_df = planned_analysis_table(
    run_sample_ids=run_sample_ids,
    variants=variants,
    scopes=SCOPES,
)
analysis_plan_df.to_csv(output_paths.table_dir / "notebook01_analysis_plan.tsv", sep="	", index=False)
display(pd.DataFrame([combined_validation]))
display(analysis_plan_df)


## Run Regression Comparison

Operation order for this CCDifference Notebook 01 run:

1. Resolve `DATA_ROOT` and set `RUN_LABEL = cellranger_filtered_manual_ec_div30_core_samples_freeze_ccdifference_v1`.
2. Load the combined normalized/log1p Notebook 00 checkpoint from `results/notebook00/cellranger_filtered_manual_ec_div30_core_samples_freeze/h5ad/manual_ec_filtered_normalized_log1p.h5ad`.
3. Validate that `.X` is normalized/log1p, `.layers["counts"]` exists and matches `.X`, and obs/var names are unique.
4. Infer the six `run_sample_id` values from the combined object, then build the analysis plan for `combined` plus each `per_sample` checkpoint.
5. For each scope/sample, score cell cycle on the normalized/log1p object using the present S and G2M genes.
6. Add `S_score`, `G2M_score`, `phase`, and `CCDifference = S_score - G2M_score` to `.obs` for that in-memory object.
7. Save cell-cycle gene-source, gene-summary, and score-summary tables for that scope/sample.
8. Select 2,000 HVGs from `.layers["counts"]` using Seurat-v3 HVG selection; these genes define the branch input feature set.
9. Run `not_regressed`: copy `adata[:, hvg_mask]`, leave `.X` as normalized/log1p HVG expression, then scale, PCA, neighbors, UMAP, and Leiden.
10. Run `regressed_ccdifference`: copy `adata[:, hvg_mask]`, run `sc.pp.regress_out(..., keys=["CCDifference"])`, then scale, PCA, neighbors, UMAP, and Leiden on the residualized `.X`.
11. Save UMAP and PCA-variance plots for both branches, colored by sample/QC/cell-cycle fields when available.
12. Save `.h5ad` output only for `regressed_ccdifference`, because this is the final branch to carry forward.
13. After all scopes finish, write the validation, HVG, cell-cycle, branch-summary, plot-manifest, and UMAP-coordinate tables.
14. Save an executed notebook copy in `results/notebook01/executed/`.

Important state detail: regression happens only on branch-specific copies after HVG subsetting. The original Notebook 00 checkpoints and their `.layers["counts"]` are not modified.


In [ ]:
input_validation_records = [combined_validation]
hvg_tables = []
hvg_parameter_tables = []
branch_records = []
plot_records = []
embedding_tables = []
cell_cycle_gene_tables = []
cell_cycle_gene_summary_tables = []
cell_cycle_score_tables = []


def safe_part(value):
    value = str(value).strip().replace(" ", "_").replace("/", "_")
    return "".join(ch for ch in value if ch.isalnum() or ch in {"_", "-", "."})


def plot_colors_for_scope(scope):
    cell_cycle_colors = ["phase", "S_score", "G2M_score", CCDIFFERENCE_KEY]
    if scope == "combined":
        return ["run_sample_id", "cell_line", "leiden", "total_counts", "pct_counts_mt", *cell_cycle_colors]
    return ["leiden", "total_counts", "pct_counts_mt", *cell_cycle_colors]


def run_scope_analysis(adata, input_path, scope, run_sample_id=None):
    validation = validate_notebook01_input(
        adata,
        input_path=input_path,
        counts_layer=input_settings.counts_layer,
        scope=scope,
        run_sample_id=run_sample_id,
    )
    if not validation["notebook01_input_validation_passed"]:
        raise AssertionError(f"Notebook 01 input validation failed: {validation}")
    input_validation_records.append(validation)

    selection = select_cell_cycle_genes(adata.var_names)
    score_cell_cycle_and_ccdifference(adata, selection=selection, ccdifference_key=CCDIFFERENCE_KEY)

    gene_table = selection.gene_table.copy()
    gene_table.insert(0, "run_sample_id", "" if run_sample_id is None else str(run_sample_id))
    gene_table.insert(0, "scope", scope)
    cell_cycle_gene_tables.append(gene_table)

    gene_summary_table = selection.summary_table.copy()
    gene_summary_table.insert(0, "run_sample_id", "" if run_sample_id is None else str(run_sample_id))
    gene_summary_table.insert(0, "scope", scope)
    cell_cycle_gene_summary_tables.append(gene_summary_table)

    cell_cycle_score_tables.append(
        cell_cycle_score_summary(adata, scope=scope, run_sample_id=run_sample_id, ccdifference_key=CCDIFFERENCE_KEY)
    )

    hvg_mask, hvg_table, hvg_params = run_hvg_selection(
        adata,
        settings=hvg_settings,
        scope=scope,
        run_sample_id=run_sample_id,
    )
    hvg_tables.append(hvg_table)
    hvg_parameter_tables.append(hvg_params)

    for variant in variants:
        branch_adata, branch_record = run_regression_embedding_branch(
            adata,
            hvg_mask=hvg_mask,
            variant=variant,
            settings=embedding_settings,
            scope=scope,
            run_sample_id=run_sample_id,
        )
        branch_records.append(branch_record)
        embedding_tables.append(
            embedding_table(
                branch_adata,
                scope=scope,
                branch=variant.branch,
                run_sample_id=run_sample_id,
            )
        )

        plot_scope_dir = output_paths.plot_dir / scope
        if scope == "per_sample":
            plot_scope_dir = plot_scope_dir / safe_part(run_sample_id)
        plot_dir = plot_scope_dir / variant.branch
        plot_dir.mkdir(parents=True, exist_ok=True)

        if SAVE_PLOTS:
            for color in plot_colors_for_scope(scope):
                if color not in branch_adata.obs.columns:
                    continue
                plot_path = plot_dir / f"umap_{safe_part(color)}.png"
                save_umap_plot(
                    branch_adata,
                    color=color,
                    output_path=plot_path,
                    title=f"{scope} {run_sample_id or ''} {variant.branch}: {color}".strip(),
                    show=SHOW_PLOTS,
                )
                plot_records.append(
                    {
                        "scope": scope,
                        "run_sample_id": "" if run_sample_id is None else str(run_sample_id),
                        "branch": variant.branch,
                        "plot_type": "umap",
                        "color": color,
                        "path": str(plot_path),
                    }
                )

            pca_path = plot_dir / "pca_variance_ratio.png"
            save_pca_variance_plot(
                branch_adata,
                output_path=pca_path,
                title=f"{scope} {run_sample_id or ''} {variant.branch}: PCA variance".strip(),
                show=SHOW_PLOTS,
            )
            plot_records.append(
                {
                    "scope": scope,
                    "run_sample_id": "" if run_sample_id is None else str(run_sample_id),
                    "branch": variant.branch,
                    "plot_type": "pca_variance_ratio",
                    "color": "",
                    "path": str(pca_path),
                }
            )

        if WRITE_BRANCH_H5AD and variant.branch in WRITE_BRANCH_H5AD_BRANCHES:
            branch_h5ad_dir = output_paths.branch_h5ad_dir(scope, variant.branch, run_sample_id=run_sample_id)
            branch_h5ad_dir.mkdir(parents=True, exist_ok=True)
            branch_path = branch_h5ad_dir / "analysis_hvg_scaled_umap.h5ad"
            branch_adata.write_h5ad(branch_path)
            branch_record["h5ad_path"] = str(branch_path)
        else:
            branch_record["h5ad_path"] = ""

        del branch_adata
        gc.collect()


if "combined" in SCOPES:
    run_scope_analysis(combined_adata, input_paths.combined_normalized_log1p, "combined")

if "per_sample" in SCOPES:
    for sample_id in run_sample_ids:
        sample_path = input_paths.per_sample_normalized_log1p(sample_id)
        sample_adata = sc.read_h5ad(sample_path)
        run_scope_analysis(sample_adata, sample_path, "per_sample", run_sample_id=sample_id)
        del sample_adata
        gc.collect()


## Save Reports


In [ ]:
input_validation_df = pd.DataFrame(input_validation_records).drop_duplicates(
    subset=["scope", "run_sample_id", "input_path"],
    keep="last",
)
hvg_genes_df = pd.concat(hvg_tables, ignore_index=True) if hvg_tables else pd.DataFrame()
hvg_parameters_df = pd.concat(hvg_parameter_tables, ignore_index=True) if hvg_parameter_tables else pd.DataFrame()
branch_summary_df = pd.DataFrame(branch_records)
plot_manifest_df = pd.DataFrame(plot_records)
umap_coordinates_df = pd.concat(embedding_tables, ignore_index=True) if embedding_tables else pd.DataFrame()
cell_cycle_gene_source_df = pd.concat(cell_cycle_gene_tables, ignore_index=True) if cell_cycle_gene_tables else pd.DataFrame()
cell_cycle_gene_summary_df = pd.concat(cell_cycle_gene_summary_tables, ignore_index=True) if cell_cycle_gene_summary_tables else pd.DataFrame()
cell_cycle_score_summary_df = pd.concat(cell_cycle_score_tables, ignore_index=True) if cell_cycle_score_tables else pd.DataFrame()

input_validation_df.to_csv(output_paths.table_dir / "notebook01_input_validation.tsv", sep="	", index=False)
hvg_genes_df.to_csv(output_paths.table_dir / "notebook01_hvg_genes.tsv", sep="	", index=False)
hvg_parameters_df.to_csv(output_paths.table_dir / "notebook01_hvg_parameters.tsv", sep="	", index=False)
branch_summary_df.to_csv(output_paths.table_dir / "notebook01_branch_summary.tsv", sep="	", index=False)
plot_manifest_df.to_csv(output_paths.table_dir / "notebook01_plot_manifest.tsv", sep="	", index=False)
umap_coordinates_df.to_csv(output_paths.table_dir / "notebook01_umap_coordinates.tsv", sep="	", index=False)
cell_cycle_gene_source_df.to_csv(output_paths.table_dir / "notebook01_cell_cycle_gene_source.tsv", sep="	", index=False)
cell_cycle_gene_summary_df.to_csv(output_paths.table_dir / "notebook01_cell_cycle_gene_summary.tsv", sep="	", index=False)
cell_cycle_score_summary_df.to_csv(output_paths.table_dir / "notebook01_cell_cycle_score_summary.tsv", sep="	", index=False)

if not input_validation_df["notebook01_input_validation_passed"].all():
    raise AssertionError("One or more Notebook 01 input validations failed.")

print("Notebook 01 output:", output_paths.run_dir)
print("Input validation rows:", len(input_validation_df))
print("HVG gene rows:", len(hvg_genes_df))
print("Cell-cycle score summary rows:", len(cell_cycle_score_summary_df))
print("Branch summary rows:", len(branch_summary_df))
print("Plot rows:", len(plot_manifest_df))
print("UMAP coordinate rows:", len(umap_coordinates_df))
display(input_validation_df)
display(branch_summary_df)
display(plot_manifest_df.head(30))


## Output Contract

Everything from this run is saved under:

`/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/notebook01/cellranger_filtered_manual_ec_div30_core_samples_freeze_ccdifference_v1/`

Already-completed cell-cycle diagnostic outputs from the earlier CCDifference PCA pass remain in the same run folder:

- `tables/cell_cycle_gene_source.tsv`
- `tables/cell_cycle_gene_summary.tsv`
- `tables/cell_cycle_genes_present.tsv`
- `tables/cell_cycle_genes_missing.tsv`
- `tables/cell_cycle_score_summary.tsv`
- `tables/cell_cycle_input_validation.tsv`
- `tables/cell_cycle_pca_diagnostic_summary.tsv`
- `tables/cell_cycle_pca_plot_manifest.tsv`
- `plots/cell_cycle_pca/combined/before_ccdifference_regression/`
- `plots/cell_cycle_pca/combined/after_ccdifference_regression/`
- `plots/cell_cycle_pca/per_sample/<run_sample_id>/before_ccdifference_regression/`
- `plots/cell_cycle_pca/per_sample/<run_sample_id>/after_ccdifference_regression/`

Main Notebook 01 tables from the CCDifference regression branch analysis are:

- `tables/notebook01_run_parameters.tsv`
- `tables/notebook01_input_validation.tsv`
- `tables/notebook01_analysis_plan.tsv`
- `tables/notebook01_hvg_genes.tsv`
- `tables/notebook01_hvg_parameters.tsv`
- `tables/notebook01_cell_cycle_gene_source.tsv`
- `tables/notebook01_cell_cycle_gene_summary.tsv`
- `tables/notebook01_cell_cycle_score_summary.tsv`
- `tables/notebook01_branch_summary.tsv`
- `tables/notebook01_plot_manifest.tsv`
- `tables/notebook01_umap_coordinates.tsv`

Main Notebook 01 plots are saved by scope and branch:

- Combined baseline branch: `plots/combined/not_regressed/`
- Combined CCDifference-regressed branch: `plots/combined/regressed_ccdifference/`
- Per-sample baseline branches: `plots/per_sample/<run_sample_id>/not_regressed/`
- Per-sample CCDifference-regressed branches: `plots/per_sample/<run_sample_id>/regressed_ccdifference/`

Useful first plots to reopen tomorrow:

- `plots/combined/not_regressed/umap_CCDifference.png`
- `plots/combined/regressed_ccdifference/umap_CCDifference.png`
- `plots/combined/regressed_ccdifference/umap_phase.png`
- `plots/combined/regressed_ccdifference/umap_leiden.png`

Final `.h5ad` outputs saved for carry-forward are only the `regressed_ccdifference` branch:

- `h5ad/combined/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-1/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-2/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-3/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-4/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-5/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`
- `h5ad/per_sample/9853-MW-6/regressed_ccdifference/analysis_hvg_scaled_umap.h5ad`

The executed notebook from the completed run is saved at:

`/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/notebook01/executed/01_notebook00_checkpoint_regression_comparison.cellranger_filtered_manual_ec_div30_core_samples_freeze_ccdifference_v1.executed.ipynb`

Completion check from this run: `notebook01_branch_summary.tsv` contains 14 branch rows: `not_regressed` and `regressed_ccdifference` for combined plus all six samples. The 7 `regressed_ccdifference` rows all have non-empty `h5ad_path` values.
